In [ ]:
#字典列表

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

import os

# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    # model="gpt-5.4-mini",
    model="deepseek-v4-flash",
    model_provider="deepseek",
    #temperature=1.5,
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    #max_tokens=15,
)

# 使用字典格式构建消息
#role:(角色)、content:(内容)、assistant: 历史回复(用于对话上下文) user:用户的输入/问题
#system:设定 AI 的行为、角色、规则
messages = [
    {"role": "system", "content": "你是一个专业的初中数学老师。"},
    {"role": "user", "content": "2 + 3 * 2 = ？"},
    {"role": "assistant", "content": "8"},
    {"role": "user", "content": "我刚才问了什么问题？答案是什么"}

]
response = model.invoke(messages)
# 打印响应
print(f"AI的回复：{response.content}")

messages2 = [
    SystemMessage("你是一位资深的计算机老师"),
    HumanMessage("1 <<= 1 = ?"),

]
print(f"AI的回复2:{model.invoke(messages2).content}")

SystemMessage {"role": "system", ...} 系统提示

HumanMessage {"role": "user", ...} 用户输入

AIMessage {"role": "assistant", ...} AI 回复

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint

messages = [
    SystemMessage(content="你是一个 Python 专家"),
    HumanMessage(content="什么是生成器？"),
]
response = model.invoke(messages)
# print(response)
# 继续对话
messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content="能给个例子吗？"))
response1 = model.invoke(messages)
rprint(response1)


AIMessage(
    content='当然可以，下面是一个更完整的实际例子，演示了生成器的几种高级用法：**使用 `send()` 
实现生产者-消费者协作、使用 `throw()` 处理异常、以及多个生成器串联形成管道**。\n\n### 
示例：基于生成器的数据处理管道\n\n我们模拟一个场景：从文本流中读取日志，过滤出错误级别的行（ERROR），再提取其中的错
误信息（假设为行中`"ERROR: "`之后的部分），最后输出前 N 条。\n\n```python\ndef read_lines(source):\n    
"""模拟从文件或字符串读取行，每次产生一行"""\n    for line in source:\n        yield line.rstrip(\'\\n\')\n\ndef 
filter_errors(lines):\n    """过滤出包含 ERROR 的行"""\n    for line in lines:\n        if \'ERROR\' in line:\n    
yield line\n\ndef extract_error_msg(lines):\n    """提取 ERROR: 后面的内容"""\n    for line in lines:\n        # 
使用 yield + send 演示：可以接收外部提供的字符串替换原错误信息\n        extra = yield line.split(\'ERROR: \', 1)[1]
if \'ERROR: \' in line else line\n        if extra:\n            # 如果外部通过 send() 
传入了额外信息，则替换当前信息\n            yield extra  # 注意：这里会多产生一个值，需谨慎设计，此处仅演示\n\ndef 
take_n(iterable, n):\n    """只取前 n 个元素"""\n    for i, item in enumerate(iterable):\n        if i >= n:\n     
break\n        yield item\n\n# 模拟数据\nlog_data = [\n    "INFO: started",\n    "ERROR: Disk full",\n    "WARNING:
memory low",\n    "ERROR: Connection timeout",\n    "INFO: retrying...",\n    "ERROR: null pointer 
exception",\n]\n\n# 构建管道\nlines = read_lines(log_data)\nerrors = filter_errors(lines)\nmsgs = 
extract_error_msg(errors)\ntop3 = take_n(msgs, 3)\n\n# 迭代并演示 send()\nfor i, msg in enumerate(top3):\n    
print(f"第 {i+1} 条错误: {msg}")\n    if i == 0:\n        # 向生成器 send 一个新值，替换下一条输出（因为 
extract_error_msg 中的第二个 yield）\n        # 注意：这会影响后续迭代，这里仅作为演示\n        try:\n            
top3.send("手动替换的错误信息")  # 尝试在迭代中 send\n        except TypeError:\n            # 在 for 循环中直接 
send 需要先预激，此处只是演示错误处理\n            print("无法在迭代中途 
send（需要从生成器外部）")\n\nprint("\\n--- 演示 throw() ---")\n# 创建一个新的生成器并手动控制\ngen = 
extract_error_msg(filter_errors(read_lines(log_data)))\ntry:\n    print(next(gen))  # 
第一条第，预激并获取第一个值\n    print(gen.send("外部注入的消息"))  # send 给第一个 yield，并获取第二个值\n    
gen.throw(ValueError, "手动触发异常")  # 在生成器内部抛出异常\nexcept StopIteration:\n    
print("生成器已结束")\nexcept ValueError as e:\n    print(f"捕获到异常: {e}")\n```\n\n### 关键点说明\n\n1. 
**惰性流水线**：`read_lines` → `filter_errors` → `extract_error_msg` → 
`take_n`，每个步骤只处理当前需要的数据，不会一次性加载全部日志。\n2. **`send()` 用法**：`extract_error_msg` 中 
`extra = yield ...` 这一行允许调用者通过 `gen.send(value)` 
向生成器内部传递值。在示例中我们演示了如何向生成器注入新数据（虽然实际使用中要小心每个 `yield` 的位置）。\n3. 
**`throw()` 用法**：可以在生成器暂停的地方注入异常，用于中断或特殊处理。\n4. **管道结束**：`take_n` 只取前 3 
条，之后整个链条停止，不会继续处理后续日志。\n\n这个例子展示了生成器在**流式处理**、**协同工作**和**外部通信**方面
的强大能力，非常适合处理大数据或实时数据流。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 
'我们被要求给一个例子。之前已经解释过生成器并提供了多个示例。但用户可能希望看到一个更具体的、实际的应用例子。我们可
以再提供一个更详细的例子，比如使用生成器处理大文件，或者实现一个生产-消费者模式，或者结合上下文管理器等。也可以进一
步解释send和close等方法。考虑用户可能需要的示例：也许是一个简单的生成器，带有交互式的send和throw演示。最好给出有实
际意义的代码，并解释运行结果。\n\n注意：用户说“能给个例子吗？”针对上面关于生成器的回答，可能是希望看到更多的示例。
我们提供一个包含send和throw的高级例子，或者一个链式生成器的例子。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 1047,
            'prompt_tokens': 829,
            'total_tokens': 1876,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 139,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 829
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-flash',
        'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
        'id': '48d72daf-adc5-4b19-b9bf-7c25384a0ada',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019f1433-517d-7342-91eb-d88ea2c55970-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 829,
        'output_tokens': 1047,
        'total_tokens': 1876,
        'input_token_details': {'cache_read': 0},
      

## 2.invoke的返回值

举例

In [16]:
response = model.invoke("用一句话解释什么是 AI")
# 1. 获取回复内容
print("AI 回复:", response.content)
# 2. 获取响应元数据
metadata = response.response_metadata
print(f"使用的模型: {metadata['model_name']}")
print(f"结束原因: {metadata['finish_reason']}")
print(f"模型提供商：{metadata['model_provider']}\n")
# 3. 获取 Token 使用情况
usage = metadata.get('token_usage', {})
print(f"输入 tokens: {usage.get('prompt_tokens')}")
print(f"输出 tokens: {usage.get('completion_tokens')}")
print(f"总计 tokens: {usage.get('total_tokens')}")
# 4. 获取消息 ID
print(f"消息 ID: {response.id}")

AI 回复: AI是让计算机系统模拟人类智能（如学习、推理、感知和决策）的技术。
使用的模型: deepseek-v4-flash
结束原因: stop
模型提供商：deepseek

输入 tokens: 9
输出 tokens: 113
总计 tokens: 122
消息 ID: lc_run--019f1436-a113-70d1-9235-b9781dd9c84c-0
